# 1. Business Problem
An Online travel booking company is suffering from loss in revenue because of the uncertain booking cancelation of its customers. The company wants to know which customer will cancel the booking.

### Dataset Detailes

0   hotel (H1 = Resort Hotel or H2 = City Hotel)

1   is_canceled Value indicating if the booking was canceled (1) or not (0)

2   lead_time Number of days that elapsed between the entering date of the booking into the PMS and the arrival date

3   arrival_date_year Year of arrival date

4   arrival_date_month Month of arrival date

5   arrival_date_week_number Week number of year for arrival date

6   arrival_date_day_of_month Day of arrival date

7   stays_in_weekend_nights Number of weekend nights (Saturday or Sunday) the guest stayed or booked to stay at the hotel

8   stays_in_week_nights Number of week nights (Monday to Friday) the guest stayed or booked to stay at the hotel

9   adults Number of adults

10  children Number of children

11  babies Number of babies

12  meal Type of meal booked. Categories are presented in standard hospitality meal packages: Undefined/SC – no meal

13  country Country of origin. Categories are represented in the ISO 3155–3:2013 format

14  market_segment Market segment designation. In categories, the term “TA” means “Travel Agents” and “TO” means “Tour Operators”

15  distribution_channel Booking distribution channel. The term “TA” means “Travel Agents” and “TO” means “Tour Operators”

16  is_repeated_guest Value indicating if the booking name was from a repeated guest (1) or not (0)

17  previous_cancellations Number of previous bookings that were cancelled by the customer prior to the current booking

18  previous_bookings_not_canceled Number of previous bookings not cancelled by the customer prior to the current booking

19  reserved_room_type Code of room type reserved. Code is presented instead of designation for anonymity reasons.

20  assigned_room_typeCode for the type of room assigned to the booking.Code is presented instead of designation for anonymity reasons.

21  booking_changes Number of changes made to the booking from the moment the booking was entered on the PMS until the moment of check-in or out

22  deposit_type Indication on if the customer made a deposit to guarantee the booking. This variable can assume three categories: No

23  agent ID of the travel agency that made the booking

24  company ID of the company that made the booking or responsible for paying the booking.

25  days_in_waiting_list Number of days the booking was in the waiting list before it was confirmed to the customer

26  customer_type Type of booking, assuming one of four categories:Transient - Transient-Party - Contract - Group

27  adr Average Daily Rate as defined by dividing the sum of all lodging transactions by the total number of staying nights

28  required_car_parking_spaces Number of car parking spaces required by the customer

29  total_of_special_requestsNumber of special requests made by the customer (e.g. twin bed or high floor)

30  reservation_status Reservation last status, assuming one of three categories: Canceled – booking was canceled by the customer; Check-Out

31  reservation_status_date Date at which the last status was set. This variable can be used in conjunction with the ReservationStatus to

# 2. Importing

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression ,RidgeClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier,StackingClassifier, ExtraTreesClassifier,AdaBoostClassifier,BaggingClassifier,VotingClassifier

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score,f1_score,classification_report,confusion_matrix
from sklearn.model_selection import GridSearchCV

from sklearn.svm import SVC
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

models = {
    "LogisticRegression": Pipeline([
        ("scaler", RobustScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    
    "RidgeClassifier": Pipeline([
        ("scaler", RobustScaler()),
        ("model", RidgeClassifier())
    ]),
    
    "KNN": Pipeline([
        ("scaler", RobustScaler()),
        ("model", KNeighborsClassifier())
    ]),
    
    "SVC": Pipeline([
        ("scaler", RobustScaler()),
        ("model", SVC())
    ]),
}

models.update({
    "DecisionTree": DecisionTreeClassifier(),
    "RandomForest": RandomForestClassifier(),
    "ExtraTrees": ExtraTreesClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "Bagging": BaggingClassifier(),
    "XGBoost": XGBClassifier(eval_metric='logloss'),
    "LightGBM": LGBMClassifier(),
    "CatBoost": CatBoostClassifier(verbose=0)
})


In [ ]:
#Load data
df=pd.read_csv("/kaggle/input/hotel-booking-demand/hotel_bookings.csv")
df.head()

In [ ]:
df.shape

# 3. Data Analysis

In [ ]:
df[df.duplicated()]

In [ ]:
summary = pd.DataFrame({
    'Non-Null Count': df.notna().sum(),
    'Non-Null (%)': (df.notna().mean() * 100).round(2),
    'Missing Count': df.isna().sum(),
    'Missing (%)': (df.isna().mean() * 100).round(2),
    'Data Type': df.dtypes
})
summary

In [ ]:
summary = pd.DataFrame({
    'Unique Values': df.nunique()
})

summary

In [ ]:
for col in df.columns:
    print(f"Unique values in column '{col}':")
    print(df[col].unique())
    print("-" * 20)

### The "company" column has more than 90% missing data and will be deleted
### The nan values from children will be filled with 0
### The nan values from agent will be filled with 0

In [ ]:
df['children'] = df['children'].fillna(0)
df['country'] = df['country'].fillna('Unknown')
df['agent'] = df['agent'].fillna(0)
df.drop(columns=['company'], inplace=True)

In [ ]:
# adults, babies and children cant be zero at same time, so dropping the rows having all these zero at same time
filter = (df.children == 0) & (df.adults == 0) & (df.babies == 0)
df[filter]

In [ ]:
is_can = len(df[df['is_canceled']==1])
print("Percentage cancelation %= ", is_can/len(df)*100)
df['reservation_status'].value_counts(normalize=True)*100

Number of guest in each hotels

# 4. Exploratory Data Analysis (EDA)

In [ ]:
counts = df['hotel'].value_counts()

bars = plt.bar(counts.index, counts.values)

plt.xlabel('Hotel Type')
plt.ylabel('Number of Guest')
plt.title('Number of Guest for Each Hotel Type')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# adaugă valori pe bare
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval, int(yval),
             ha='center', va='bottom')

plt.show()

In [ ]:
country_wise_counts = df[df['is_canceled'] == 0]['country'].value_counts().reset_index()
country_wise_counts.columns = ['country', 'No of guests']
country_wise_counts

In [ ]:
basemap = folium.Map()
guests_map = px.choropleth(country_wise_counts, locations = country_wise_counts['country'],
                           color = country_wise_counts['No of guests'], hover_name = country_wise_counts['country'])
guests_map.show()

Most guests are from Portugal and other countries in Europe.

In [ ]:
dfx = df[df['is_canceled'] == 0]
px.box(data_frame = dfx, x = 'reserved_room_type', y = 'adr', color = 'hotel', template = 'plotly_dark')

In [ ]:
month_order = [
    'January','February','March','April','May','June',
    'July','August','September','October','November','December'
]

top_months = df["arrival_date_month"].value_counts().reindex(month_order)

plt.figure(figsize=(10,5))

sns.barplot(
    x=top_months.index,
    y=top_months.values,
    hue=top_months.index,
    palette='muted',
    legend=False
)

plt.title("Bookings by Month")
plt.xlabel("Month")
plt.ylabel("Number of Bookings")

plt.xticks(rotation=45)
plt.show()

Top booking months: August, July, May → peak summer vacation season¶
Least bookings: November, February → off-season periods

Shows clear seasonal booking trends in hotels

In [ ]:
counts = df.groupby(['arrival_date_year', 'is_canceled']).size().unstack()
counts = counts.reindex(columns=[0, 1], fill_value=0)

x = np.arange(len(counts.index))
width = 0.35

plt.figure(figsize=(7,4))

bars1 = plt.bar(x - width/2, counts[0], width, label='Not Canceled')
bars2 = plt.bar(x + width/2, counts[1], width, label='Canceled')

# valori deasupra barelor
for bar in bars1:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f'{int(height)}',
        ha='center',
        va='bottom'
    )

for bar in bars2:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f'{int(height)}',
        ha='center',
        va='bottom'
    )

plt.xticks(x, counts.index)

plt.title('Bookings per Year')
plt.xlabel('Year')
plt.ylabel('Number of Bookings')

plt.legend()

plt.show()

In [ ]:
df_resort = df[(df['hotel'] == 'Resort Hotel') & (df['is_canceled'] == 0)]
df_city = df[(df['hotel'] == 'City Hotel') & (df['is_canceled'] == 0)]

In [ ]:
resort_hotel = df_resort.groupby(['arrival_date_month'])['adr'].mean().reset_index()
resort_hotel

In [ ]:
city_hotel = df_city.groupby(['arrival_date_month'])['adr'].mean().reset_index()
city_hotel

In [ ]:
final_hotel = resort_hotel.merge(city_hotel, on = 'arrival_date_month')
final_hotel.columns = ['month', 'price_for_resort', 'price_for_city_hotel']
final_hotel

In [ ]:
month_order = [
    'January', 'February', 'March', 'April', 'May', 'June',
    'July', 'August', 'September', 'October', 'November', 'December'
]

final_hotel['month'] = pd.Categorical(
    final_hotel['month'],
    categories=month_order,
    ordered=True
)

final_prices = final_hotel.sort_values('month')
final_prices

In [ ]:
plt.figure(figsize = (19, 10))

px.line(final_prices, x = 'month', y = ['price_for_resort','price_for_city_hotel'],
        title = 'Room price per night over the Months', template = 'plotly_dark')

Resort_Hotel- expensive in augest month  //  
City_Hotel- expensive in may month

In [ ]:
df2 = df[df["is_canceled"] == 0].copy()
df2.head()

In [ ]:
df2['total_night']=df2['stays_in_weekend_nights']+df2['stays_in_week_nights']
df2

In [ ]:
stay = df2.groupby(['total_night', 'hotel']).agg('count').reset_index()
stay = stay.iloc[:, :3]
stay = stay.rename(columns={'is_canceled':'Number of stays'})
stay

In [ ]:
px.bar(data_frame = stay, x = 'total_night', y = 'Number of stays', color = 'hotel', barmode = 'group',
        template = 'plotly_dark')

In [ ]:
plt.figure(figsize=(10,8))
sns.barplot(x=df[df['is_canceled']==0].groupby('market_segment')['stays_in_weekend_nights'].count().index,
            y=df[df['is_canceled']==0].groupby('market_segment')['stays_in_weekend_nights'].count())

# 5. Data Preprocessing

### DROP unneeded_columns

In [ ]:
cols_to_drop = [
    'reservation_status',
    'reservation_status_date',
    'agent',
    'arrival_date_day_of_month',
    'arrival_date_week_number'
]

df.drop(columns=cols_to_drop, inplace=True)

### Data_encoding

In [ ]:
categorical_cols = [
    'hotel',
    'arrival_date_month',
    'meal',
    'country',
    'market_segment',
    'distribution_channel',
    'reserved_room_type',
    'assigned_room_type',
    'deposit_type',
    'customer_type',
    
]
df = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True
)

### Data_split

In [ ]:
x=df.drop("is_canceled",axis=1)
y=df["is_canceled"]


x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=.2)

### Class Imbalance Treatment using SMOTE
### SMOTE is applied to the training set to balance booking cancellation classes and enhance the model's predictive performance.

In [ ]:
from imblearn.over_sampling import SMOTE
sm=SMOTE(random_state=42)
x_train_res,y_train_res=sm.fit_resample(x_train,y_train)

### Antrenare + evaluare

In [ ]:
results = []

for name, model in models.items():
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    })


results_df = pd.DataFrame(results).sort_values(by="F1 Score", ascending=False)

results_df

### Comparare rezultate

In [ ]:
results_df = pd.DataFrame(results)
results_df.sort_values(by="F1 Score", ascending=False)

### Raport detaliat pentru fiecare model

In [ ]:
for name, model in models.items():
    print(f"\n{name}")
    print(classification_report(y_test, model.predict(x_test)))

### Confusion matrix

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, models["RandomForest"].predict(X_test))

sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix - RandomForest")
plt.show()

# 6. Conclusion – Hotel Booking Cancellation Prediction 📊🏨

After evaluating 8 different models, the performance comparison highlights the following insights:

CatBoost, XGBoost, and LightGBM achieved the best balance between Test Accuracy and Recall for canceled bookings (Class 1).

Extra Trees provided the highest recall (0.74), maximizing the detection of cancellations, but at the cost of lower precision and overall accuracy.

Logistic Regression performed the lowest but serves as a strong baseline for comparison.

Random Forest and AdaBoost delivered good balanced performance, suitable when both cancellation detection and overall accuracy are important.

Decision Tree is decent but lags behind RF, XGBoost, and CatBoost.

Final Ranking (Best Overall Performance)

CatBoost

XGBoost

LightGBM

AdaBoost

Random Forest

Decision Tree

Extra Trees (best recall for cancellations)

Logistic Regression (baseline)

💡 Insight:

If the main goal is maximizing detected cancellations, Extra Trees or a Voting ensemble combining top Tree-based models is recommended.

For a balanced performance in both accuracy and cancellation detection, CatBoost, XGBoost, or LightGBM are optimal choices.